In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import shap
import joblib
from sklearn.metrics import mean_squared_error

c:\Users\celos\OneDrive\Documentos\VSCode\hack\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
modelo = joblib.load("modelo_treinado.pkl")

In [ ]:
# 1) Carregamento (ajuste o caminho e o tipo)
df = pd.read_csv("base_teste.csv")  # se estiver em CSV
df = df.dropna()
df.reset_index(drop=True, inplace=True)

# 2) Dividir os dados em conjunto de treinamento e teste
X = df[['product', 'processada_ml']]  # Features
y = df.escala2  # Target

y_pred = modelo.predict(X)

mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)

print(f'Erro quadrático médio (MSE): {mse:.2f}')
print(f'Raiz do erro quadrático médio (RMSE): {rmse:.2f}')


Erro quadrático médio (MSE): 0.04
Raiz do erro quadrático médio (RMSE): 0.20


In [7]:
df.head()

,product,processada_ml,escala
0,"Money transfer, virtual currency, or money ser...",complete service client pay 400 via fully comp...,-0.75
1,"Money transfer, virtual currency, or money ser...",complete service client pay 400 via fully comp...,-0.75
2,Credit reporting or other personal consumer re...,write formally file complaint financial believ...,-0.30
3,"Payday loan, title loan, personal loan, or adv...",happen dispute several charge paypal credit ac...,-0.45
4,Checking or savings account,year open check account keybank downtown branc...,-0.45


In [8]:
df['escala_predita'] = y_pred
df.head()

,product,processada_ml,escala,escala_predita
0,"Money transfer, virtual currency, or money ser...",complete service client pay 400 via fully comp...,-0.75,-0.370892
1,"Money transfer, virtual currency, or money ser...",complete service client pay 400 via fully comp...,-0.75,-0.370892
2,Credit reporting or other personal consumer re...,write formally file complaint financial believ...,-0.30,-0.267548
3,"Payday loan, title loan, personal loan, or adv...",happen dispute several charge paypal credit ac...,-0.45,-0.357081
4,Checking or savings account,year open check account keybank downtown branc...,-0.45,-0.339401


In [9]:
labels = ['muito negativo', 'negativo', 'neutro', 'positivo']
valores = [-1, -0.5, 0, 0.5, 1]
pd.cut(df['escala'], bins=valores, labels=labels).value_counts()

escala
negativo          31982
muito negativo    27780
neutro            12242
positivo             13
Name: count, dtype: int64

In [10]:
labels = ['muito negativo', 'negativo', 'neutro', 'positivo']
valores = [-1, -0.5, 0, 0.5, 1]
pd.cut(df['escala_predita'], bins=valores, labels=labels).value_counts()

escala_predita
negativo          62097
muito negativo     8492
neutro             1427
positivo              0
Name: count, dtype: int64

In [ ]:
print(classification_report(y_test, pred))
print("Matriz de confusão:\n", confusion_matrix(y_test, pred))

In [11]:
# =========================================
# 2) Nomes das features (TF-IDF)
# =========================================
feature_names = vectorizer.get_feature_names_out()  # ordem = colunas do X_tfidf

# =========================================
# 3) Importâncias do XGBoost (sem sinal)
# =========================================
# Para XGBRegressor/XGBClassifier do xgboost.sklearn
xgb_importances = model.feature_importances_  # shape: (n_features,)

# Top-k termos por importância
k = 50
top_idx = np.argsort(xgb_importances)[::-1][:k]
top_terms = feature_names[top_idx]
top_scores = xgb_importances[top_idx]

# Plot de barras dos top termos (opcional)
plt.figure(figsize=(8, 10))
ypos = np.arange(len(top_terms))
plt.barh(ypos, top_scores[::-1], color="steelblue")
plt.yticks(ypos, top_terms[::-1])
plt.gca().invert_yaxis()
plt.title("Top termos por importância do XGBoost (global, sem sinal)")
plt.xlabel("Importância")
plt.tight_layout()
plt.show()

# =========================================
# 4) SHAP: contribuições com sinal
# =========================================
# Para conjuntos muito grandes, amostre para acelerar
sample_size = min(3000, X_tfidf.shape[0])
rng = np.random.default_rng(42)
sample_idx = rng.choice(X_tfidf.shape[0], size=sample_size, replace=False)

X_sample = X_tfidf[sample_idx]

# Criar o Explainer (TreeExplainer para XGBoost)
explainer = shap.Explainer(model)  # shap auto-detecta booster
shap_values = explainer(X_sample)  # valores SHAP (por amostra x feature)

# shap_values.values -> matriz (n_amostras, n_features)
sv = shap_values.values

# Contribuição média por termo (com sinal)
mean_shap = sv.mean(axis=0)  # shape: (n_features,)

# Para nuvem de palavras positiva (valores médios positivos)
pos_mask = mean_shap > 0
pos_scores = mean_shap[pos_mask]
pos_terms = feature_names[pos_mask]

# Para nuvem de palavras negativa (valores médios negativos)
neg_mask = mean_shap < 0
neg_scores = -mean_shap[neg_mask]  # usa magnitude para a nuvem
neg_terms = feature_names[neg_mask]

# =========================================
# 5) Nuvem de palavras (WordCloud)
#    - "Geral": com importâncias do XGBoost (não tem sinal)
#    - "Positiva": termos com contribuição média positiva (SHAP)
#    - "Negativa": termos com contribuição média negativa (SHAP)
# =========================================
def plot_wordcloud_from_terms_scores(terms, scores, title, max_words=150, colormap="viridis"):
    # Cria dicionário {termo: peso}
    freqs = {t: float(s) for t, s in zip(terms, scores)}
    wc = WordCloud(width=1000, height=600,
                   background_color="white",
                   colormap=colormap,
                   max_words=max_words,
                   prefer_horizontal=0.95)
    wc.generate_from_frequencies(freqs)
    plt.figure(figsize=(12, 7))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Nuvem geral (importâncias globais)
plot_wordcloud_from_terms_scores(feature_names, xgb_importances,
                                 title="Nuvem de Palavras (Importância XGBoost - Global, sem sinal)",
                                 colormap="cividis")

# Nuvem positiva (termos que puxam a predição para cima)
plot_wordcloud_from_terms_scores(pos_terms, pos_scores,
                                 title="Nuvem de Palavras Positivas (Contribuição SHAP média > 0)",
                                 colormap="Greens")

# Nuvem negativa (termos que puxam a predição para baixo)
plot_wordcloud_from_terms_scores(neg_terms, neg_scores,
                                 title="Nuvem de Palavras Negativas (Contribuição SHAP média < 0)",
                                 colormap="Reds")

NameError: name 'vectorizer' is not defined